<a href="https://colab.research.google.com/github/Graciliana/Heart-disease-prediction/blob/main/03_modeling.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Modelagem


# 03 — Modelagem e Validação

## Objetivo

Nesta etapa serão desenvolvidos e comparados modelos de Machine Learning
para classificação da variável alvo `HeartDisease`.

Serão utilizados obrigatoriamente os algoritmos:

- K-Nearest Neighbors (KNN);
- Decision Tree.

Como experimento adicional, será utilizado o XGBoost como benchmark.

Para o KNN serão avaliados diferentes valores de `n_neighbors`.
Para a Decision Tree serão avaliados diferentes valores de `max_depth`.

As métricas serão calculadas tanto no conjunto de treinamento quanto
no conjunto de teste, permitindo avaliar a capacidade de generalização
dos modelos e identificar possíveis sinais de overfitting.

A seleção dos modelos será realizada considerando principalmente o
desempenho no conjunto de teste e o equilíbrio entre treinamento e teste.

## 2. Importação das Bibliotecas

In [ ]:
import warnings

warnings.filterwarnings("ignore")

import joblib
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

from pathlib import Path

from sklearn.neighbors import KNeighborsClassifier
from sklearn.neighbors import NearestNeighbors

from sklearn.tree import DecisionTreeClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

from xgboost import XGBClassifier

## 3. Configuração

In [ ]:

RANDOM_STATE = 42

JADE = "#00A93F"

pd.set_option(
    "display.max_columns",
    None
)

## 4. Carregamento dos Dados

In [15]:

# Dados para KNN
X_test_knn = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/sctech/Heart-disease-prediction/data/processed/X_test_knn.csv")

X_train_knn = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/sctech/Heart-disease-prediction/data/processed/X_train_knn.csv")

y_train_knn = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/sctech/Heart-disease-prediction/data/processed/y_train_knn.csv").squeeze()

# Dados para Decision Tree e XGBoost
X_train_tree = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/sctech/Heart-disease-prediction/data/processed/X_train_tree.csv")

X_test_tree = pd.read_csv(   "/content/drive/MyDrive/Colab Notebooks/sctech/Heart-disease-prediction/data/processed/X_test_tree.csv")

y_train_tree = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/sctech/Heart-disease-prediction/data/processed/y_train_tree.csv").squeeze()

# teste


y_test = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/sctech/Heart-disease-prediction/data/processed/y_test.csv").squeeze()

## 5. Validação dos dados

In [ ]:
# @title Verificar as dimensões
print("=" * 60)
print("DIMENSÕES DOS DATASETS")
print("=" * 60)

print(f"X_train_knn : {X_train_knn.shape}")
print(f"y_train_knn : {y_train_knn.shape}")

print(f"X_test_knn  : {X_test_knn.shape}")
print(f"y_test      : {y_test.shape}")

print()

print(f"X_train_tree: {X_train_tree.shape}")
print(f"y_train_tree: {y_train_tree.shape}")

print(f"X_test_tree : {X_test_tree.shape}")
print(f"y_test      : {y_test.shape}")

DIMENSÕES DOS DATASETS
X_train_knn : (439128, 50)
X_test_knn  : (60344, 50)
X_train_tree: (439128, 50)
X_test_tree : (60344, 50)
y_train     : (241373,)
y_test      : (60344,)


In [ ]:
# @title Verificação de valores ausentes
missing_summary = pd.DataFrame({
    "Dataset": [
        "X_train_knn",
        "X_test_knn",
        "X_train_tree",
        "X_test_tree",
        "y_train",
        "y_test"
    ],

    "Missing Values": [
        X_train_knn.isna().sum().sum(),
        X_test_knn.isna().sum().sum(),
        X_train_tree.isna().sum().sum(),
        X_test_tree.isna().sum().sum(),
        y_train.isna().sum(),
        y_test.isna().sum()
    ]
})

display(missing_summary)

,Dataset,Missing Values
0,X_train_knn,0
1,X_test_knn,0
2,X_train_tree,0
3,X_test_tree,0
4,y_train,0
5,y_test,0


## 6. Função de avaliação

In [16]:
def evaluate_model(
    model,
    X_train,
    y_train,
    X_test,
    y_test
):
    """
    Treina um modelo de classificação e calcula métricas
    nos conjuntos de treinamento e teste.

    Parameters
    ----------
    model : sklearn estimator
        Modelo de classificação a ser treinado.

    X_train : pandas.DataFrame
        Variáveis preditoras do conjunto de treinamento.

    y_train : pandas.Series
        Variável alvo do conjunto de treinamento.

    X_test : pandas.DataFrame
        Variáveis preditoras do conjunto de teste.

    y_test : pandas.Series
        Variável alvo do conjunto de teste.

    Returns
    -------
    model : sklearn estimator
        Modelo treinado.

    dict
        Dicionário contendo as métricas de treinamento,
        teste e diferença entre Train F1 e Test F1.
    """

    # ==============================
    # Treinamento
    # ==============================

    model.fit(
        X_train,
        y_train
    )

    # ==============================
    # Predições
    # ==============================

    y_train_pred = model.predict(
        X_train
    )

    y_test_pred = model.predict(
        X_test
    )

    # ==============================
    # Métricas
    # ==============================

    metrics = {
        "Train Accuracy": accuracy_score(
            y_train,
            y_train_pred
        ),

        "Test Accuracy": accuracy_score(
            y_test,
            y_test_pred
        ),

        "Train Precision": precision_score(
            y_train,
            y_train_pred,
            zero_division=0
        ),

        "Test Precision": precision_score(
            y_test,
            y_test_pred,
            zero_division=0
        ),

        "Train Recall": recall_score(
            y_train,
            y_train_pred,
            zero_division=0
        ),

        "Test Recall": recall_score(
            y_test,
            y_test_pred,
            zero_division=0
        ),

        "Train F1": f1_score(
            y_train,
            y_train_pred,
            zero_division=0
        ),

        "Test F1": f1_score(
            y_test,
            y_test_pred,
            zero_division=0
        )
    }

    # ==============================
    # Gap entre treino e teste
    # ==============================

    metrics["F1 Gap"] = (
        metrics["Train F1"]
        -
        metrics["Test F1"]
    )

    return model, metrics

## 7. Modelagem - **KNN**



In [ ]:
K_VALUES = [3, 5, 7, 9]

In [ ]:
# @title ## Treinamento
knn_results = []

knn_models = {}

for k in K_VALUES:

    print(f"Treinando KNN | K = {k}")

    model = KNeighborsClassifier(
        n_neighbors=k,
        n_jobs=-1
    )

    model, metrics = evaluate_model(
        model=model,
        X_train=X_train_knn,
        y_train=y_train_knn,
        X_test=X_test_knn,
        y_test=y_test
    )

    metrics["Model"] = "KNN"
    metrics["n_neighbors"] = k

    knn_results.append(metrics)

    knn_models[k] = model

Treinando KNN | K = 3
